# 04_make_fingerprints — fingerprint 6종 계산

**한 줄 요약:** `same_dedup_keepdiff` 시트 분자들에 대해 **6가지 fingerprint**를 계산해 종류별 시트로 저장한다.
**6종:** ECFP4(원자주변 원형)·MACCS(166 규칙)·RDKit(경로)·AtomPair(원자쌍) + **Avalon**·**TopologicalTorsion**(원자 3개 비틀림).
**큰 흐름:** ① 준비·읽기 → ② 유효 분자만 → ③ 생성기 6종 → ④ 계산·저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]. ③은 처음 나온 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)* `os.chdir('..')`=상위 폴더 이동.

### 셀 1 — 도구 + 데이터 읽기
fingerprint 라이브러리(Avalon 포함)를 가져오고 3번째 시트를 읽는다.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit.Avalon import pyAvalonTools           # ← 추가 지문(Avalon)용
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SRC = "data/HSD17B13_IC50_merged.xlsx"
OUT = "data/HSD17B13_fingerprints.xlsx"
NBITS = 1024
df = pd.read_excel(SRC, sheet_name="same_dedup_keepdiff")

🔎 **코드 뜯어보기 (셀 1)**
- `from rdkit.Avalon import pyAvalonTools` : **Avalon** 지문 계산 도구(추가된 6번째 지문). `from rdkit import Chem, DataStructs`=분자·지문 변환. `NBITS=1024`=지문 길이.

### 셀 2 — 기준 열 + 유효 분자만
결과에 붙일 정보 열을 정하고 SMILES 있는 행만 남긴다.

In [ ]:
# fingerprint 계산에 쓸 기준 컬럼 (구조 = canonical_smiles, 라벨 = ic50_nM)
meta_cols = ["canonical_smiles", "ic50_nM", "relation", "sources"]
df = df.dropna(subset=["canonical_smiles"]).reset_index(drop=True)

🔎 **코드 뜯어보기 (셀 2)**
- `meta_cols=[...]` : 결과에 함께 넣을 정보 열. `.dropna(subset=[...]).reset_index(drop=True)`=SMILES 없는 행 제거 후 번호 새로.

### 셀 3 — 지문 생성기 6종 + 변환 함수
4개 생성기 + TopologicalTorsion 생성기, 그리고 MACCS·Avalon을 배열로 바꾸는 함수를 만든다.

In [ ]:
# 생성기(한 번만 만들어 재사용) — 6종
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
gen_rdk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NBITS)
gen_ap = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NBITS)
gen_tt = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=NBITS)   # ← 추가

def maccs_np(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def avalon_np(mol):                                # ← 추가
    fp = pyAvalonTools.GetAvalonFP(mol, NBITS)
    arr = np.zeros((NBITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

🔎 **코드 뜯어보기 (셀 3)** *(Morgan/RDKit/AtomPair 생성기는 그대로)*
- `GetTopologicalTorsionGenerator(fpSize=NBITS)` : **원자 4개가 이루는 비틀림(torsion)** 패턴 지문 생성기(추가).
- `def avalon_np(mol):` : Avalon 지문을 numpy 배열로. `pyAvalonTools.GetAvalonFP(mol, NBITS)`=Avalon 지문 계산, `DataStructs.ConvertToNumpyArray(fp, arr)`=0/1 배열로 채우기(MACCS와 같은 방식).

### 셀 4 — 분자마다 6종 지문 계산 → 시트별 저장
분자를 한 번씩 읽어 6가지 지문을 동시에 계산하고 종류별 시트로 저장한다.

In [ ]:
# 분자 파싱은 한 번만, 6가지 FP 동시 계산
rows = {"ECFP4": [], "MACCS": [], "RDKit": [], "AtomPair": [], "Avalon": [], "TopoTorsion": []}
keep_idx = []
for i, smi in enumerate(df["canonical_smiles"]):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    keep_idx.append(i)
    rows["ECFP4"].append(gen_ecfp.GetFingerprintAsNumPy(mol))
    rows["RDKit"].append(gen_rdk.GetFingerprintAsNumPy(mol))
    rows["AtomPair"].append(gen_ap.GetFingerprintAsNumPy(mol))
    rows["TopoTorsion"].append(gen_tt.GetFingerprintAsNumPy(mol))
    rows["MACCS"].append(maccs_np(mol))
    rows["Avalon"].append(avalon_np(mol))

meta = df.loc[keep_idx, meta_cols].reset_index(drop=True)
with pd.ExcelWriter(OUT, engine="openpyxl") as w:
    for name, mat in rows.items():
        X = np.vstack(mat)
        cols = [f"X{j+1}" for j in range(X.shape[1])]
        fp_df = pd.concat([meta, pd.DataFrame(X, columns=cols)], axis=1)
        fp_df.to_excel(w, sheet_name=name, index=False)
        print(f"[{name}] {X.shape[0]}행 x {X.shape[1]}bit")
print("\n저장 완료 →", OUT)
print(f"입력 {len(df)}개 중 {len(keep_idx)}개 정상 변환")

🔎 **코드 뜯어보기 (셀 4)** *(loop·vstack·ExcelWriter는 이전에 설명)*
- `rows` 딕셔너리에 지문 종류 **6개**로 확장. 각 분자마다 6개 생성기/함수로 지문을 계산해 해당 리스트에 추가. `for name, mat in rows.items():`로 종류별 시트 저장.